# Custom loss · Chunked cross-entropy

Train Qwen3.5-2B on JOSIE with a loss defined in this notebook.
Replace `trainer.task.loss`; the normal training, validation, and saving pipeline stays in use.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
from collections.abc import Mapping
from dataclasses import replace
from functools import partial
from pathlib import Path

import mlx.core as mx
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, prepare_model_for_training
from mlx_vlm.trainer.losses.cross_entropy import cross_entropy
from mlx_vlm.trainer.vlm.sft.runtime import forward_vlm_logits, vlm_batch_metrics

## 1. Settings

Edit the model, data, and output path below. `iters` counts batches;
four accumulation steps make one optimizer update. Paths use the kernel's working directory.

Set sample limits to `None` to use the entire partition.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/JOSIE-v2-Instruct-5K"
VALIDATION_FRACTION = 0.05
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
CHUNK_SIZE = 128
OUTPUT_DIR = Path("adapters/vlm/custom_loss")
SEED = 42

# Adapters
TRAIN_TYPE = "lora"  # "lora", "dora", or "full"
CHECKPOINT_PATH = None
LORA_RANK = 8
LORA_ALPHA = 16

# Training
args = VLMTrainingArgs(
    iters=100,
    batch_size=1,
    learning_rate=1e-4,
    gradient_accumulation_steps=4,
    max_seq_length=2048,
    train_on_completions=True,
    cache_size=8,
    grad_checkpoint=True,
    compile=False,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()

## 2. Data

Load [JOSIE instructions](https://huggingface.co/datasets/mlx-community/JOSIE-v2-Instruct-5K),
keep conversations with a non-empty final assistant response, and create a seeded
training/validation split.


In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
if "messages" not in dataset.column_names:
    raise ValueError(f"Expected a messages column, got {dataset.column_names}.")


def has_assistant_target(row):
    messages = row["messages"]
    return bool(
        messages
        and messages[-1].get("role") == "assistant"
        and isinstance(messages[-1].get("content"), str)
        and messages[-1]["content"].strip()
    )


source_rows = len(dataset)
dataset = dataset.filter(
    has_assistant_target, desc="Keep conversations with assistant targets"
)
dropped_rows = source_rows - len(dataset)
print({"source_rows": source_rows, "dropped_without_assistant_target": dropped_rows})
if len(dataset) < 2:
    raise ValueError("Need at least two conversations with assistant targets to split.")

partitions = dataset.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Model

Load a fresh base model before preparing its adapters.


In [ ]:
model, processor = load(MODEL_ID)

### Prepare adapters

LoRA trains the language adapters while freezing the base. `TRAIN_TYPE` also accepts
`"dora"` or `"full"`; full training unfreezes all weights and needs more memory.
Set `CHECKPOINT_PATH` to saved weights to continue with a fresh optimizer.
See the [training guide](../../docs/training.md) for quantization and target-layer options.


In [ ]:
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    checkpoint_path=CHECKPOINT_PATH,
)

## 4. Custom loss

Return `loss, metrics` using MLX arrays. `weight` controls averaging; `num_tokens`
counts supervised targets. Other scalar metrics are logged automatically.

Reduce cross-entropy over token chunks, sum their losses, then divide by the total
target count. Checkpoint each chunk's intermediate computations. The model still
allocates full `[batch, time, vocabulary]` logits; this does not chunk its output projection.


In [ ]:
@mx.checkpoint
def nll_chunk_sum(logits, targets, mask):
    """Sum one token chunk's masked NLL, accumulating in float32."""
    logits = logits.astype(mx.float32)
    targets = mx.stop_gradient(targets)
    mask = mx.stop_gradient(mask)
    target_logits = mx.take_along_axis(logits, targets[:, None], axis=-1)[:, 0]
    nll = mx.logsumexp(logits, axis=-1) - target_logits
    return mx.where(mask, nll, 0.0).sum()


def chunked_nll(logits, targets, mask=None, *, chunk_size=128, ignore_index=-100):
    """Return (mean supervised-token NLL, scalar metrics).

    Inputs are aligned logits [batch, time, vocab] and targets [batch, time].
    Causal shifting belongs to the caller. Selected labels must be valid
    vocabulary IDs; masked/ignored labels are replaced before gathering.
    """
    if not isinstance(chunk_size, int) or chunk_size < 1:
        raise ValueError("chunk_size must be a positive integer.")
    if logits.ndim != 3 or targets.ndim != 2 or logits.shape[:-1] != targets.shape:
        raise ValueError("Expected aligned logits [B, T, V] and targets [B, T].")
    if mask is None:
        mask = mx.ones(targets.shape, dtype=mx.bool_)
    elif mask.shape != targets.shape:
        raise ValueError("mask must have the same shape as targets.")
    else:
        mask = mask.astype(mx.bool_)
    if ignore_index is not None:
        mask = mx.logical_and(mask, targets != ignore_index)

    safe_targets = mx.where(mask, targets, 0).reshape(-1)
    flat_mask = mask.reshape(-1)
    flat_logits = logits.reshape(-1, logits.shape[-1])
    count = flat_mask.sum()
    total = mx.array(0.0, dtype=mx.float32)
    for start in range(0, flat_logits.shape[0], chunk_size):
        stop = start + chunk_size
        total = total + nll_chunk_sum(
            flat_logits[start:stop], safe_targets[start:stop], flat_mask[start:stop]
        )
    denominator = mx.maximum(count.astype(mx.float32), 1.0)
    loss = total / denominator
    correct = mx.logical_and(
        mx.argmax(logits, axis=-1) == safe_targets.reshape(targets.shape), mask
    ).sum()
    return loss, {
        "weight": count,
        "num_tokens": count,
        "nll": loss,
        "token_accuracy": correct.astype(mx.float32) / denominator,
    }

### Check correctness

Compare losses and gradients with the shared cross-entropy on nine small cases,
including ignored labels, uneven chunks, and empty supervision.


In [ ]:
logits = mx.random.normal((2, 5, 8))
targets = mx.array([[1, -100, 3, -999, 5], [0, 1, 2, 3, 4]])
mask = mx.array([[True, True, True, False, True], [True, False, True, True, True]])

for labels, selected in (
    (targets, mask),
    (mx.full(targets.shape, -100, dtype=mx.int32), None),
    (targets, mx.zeros(targets.shape, dtype=mx.bool_)),
):
    expected_loss, expected_metrics = cross_entropy(logits, labels, selected)
    expected_grad = mx.grad(lambda x: cross_entropy(x, labels, selected)[0])(logits)
    for size in (1, 3, 128):
        actual_loss, actual_metrics = chunked_nll(
            logits, labels, selected, chunk_size=size
        )
        actual_grad = mx.grad(
            lambda x: chunked_nll(x, labels, selected, chunk_size=size)[0]
        )(logits)
        mx.eval(actual_loss, actual_metrics, actual_grad, expected_loss, expected_grad)
        assert (
            actual_metrics["num_tokens"].item() == expected_metrics["num_tokens"].item()
        )
        assert mx.allclose(
            actual_metrics["token_accuracy"], expected_metrics["token_accuracy"]
        ).item()
        assert mx.allclose(actual_loss, expected_loss, atol=1e-6, rtol=1e-5).item()
        assert mx.allclose(actual_grad, expected_grad, atol=1e-6, rtol=1e-5).item()

assert mx.all(expected_grad == 0).item()
print("Loss, token-count, and gradient parity passed for nine cases.")

### Connect the model forward

Shift targets for next-token prediction and apply the attention/completion masks.
Keep media statistics in the metrics dictionary so the same loss can support vision rows.


In [ ]:
def chunked_vlm_sft_loss(
    model, batch: Mapping, *, chunk_size=128, train_on_completions=False
):
    """Use the usual VLM forward with a notebook-defined chunked NLL."""
    ids = batch["input_ids"]
    attention = batch["attention_mask"]
    if ids.ndim != 2 or ids.shape[1] < 2 or attention.shape != ids.shape:
        raise ValueError("Expected input_ids/attention_mask [B, T] with T >= 2.")
    targets = ids[:, 1:]
    mask = attention[:, 1:].astype(mx.bool_)
    if train_on_completions:
        completion = batch.get("completion_mask")
        if completion is None or completion.shape != ids.shape:
            raise ValueError(
                "Completion-only loss needs completion_mask aligned with input_ids."
            )
        mask = mx.logical_and(mask, completion[:, 1:].astype(mx.bool_))
    logits = forward_vlm_logits(model, ids[:, :-1], attention[:, :-1], batch)
    loss, metrics = chunked_nll(logits, targets, mask, chunk_size=chunk_size)
    return loss, {**metrics, **vlm_batch_metrics(batch, model)}

## 5. Train

Build the standard SFT trainer, then replace its loss before training.
Both training and validation call the custom objective.


In [ ]:
trainer = Trainer(
    model,
    args,
    task="vlm",
    algorithm="sft",
    processing_class=processor,
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    seed=SEED,
)

trainer.task = replace(
    trainer.task,
    loss=partial(
        chunked_vlm_sft_loss,
        chunk_size=CHUNK_SIZE,
        train_on_completions=trainer.args.train_on_completions,
    ),
)

In [ ]:
metrics = trainer.train()

### Metrics

The table shows the last five training reports. All metrics remain in `metrics`
and `trainer.log_history`, including loss diagnostics, timing, and memory.
`trained_tokens` counts supervised targets; `total_vision_tokens` counts expanded
image/video input positions. Processing token/s includes forward, backward, and updates.


In [ ]:
report_history = pd.DataFrame(trainer.log_history)
columns = {
    "iteration": "step",
    "train_loss": "loss",
    "learning_rate": "lr",
    "trained_tokens": "trained tokens",
    "total_vision_tokens": "vision tokens",
    "processed_tokens_per_second": "processed tok/s",
    "peak_memory": "peak GB",
}
recent_reports = (
    report_history.dropna(subset=["train_loss"])
    .tail(5)
    .reindex(columns=columns)
    .rename(columns=columns)
    .set_index("step")
)
recent_reports.round({"loss": 4, "processed tok/s": 1, "peak GB": 2})

## 6. Evaluate

Check the custom objective on held-out conversations.


In [ ]:
assert mx.isfinite(mx.array(metrics["train_loss"])).item(), "Non-finite training loss."
validation = trainer.evaluate() if eval_rows is not None else {}
pd.DataFrame([validation]).filter(
    items=["val_loss", "val_token_accuracy", "val_perplexity", "val_num_vision_tokens"]
).round(4)

## 7. Save

Keep `weights.safetensors` and `adapter_config.json` together. The loop also saves
the final checkpoint automatically.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())

### Reload later

Load the original base model, then attach the saved weights:

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
model.eval()
```

More options: [training guide](../../docs/training.md).

Keep this notebook or your custom-loss package to reconstruct the objective;
weight checkpoints do not serialize the callable.
